# Pertemuan 10 · Data Preprocessing & Feature Engineering
**Studi kasus PBL: memprediksi mahasiswa yang lulus tepat waktu**

Bagian akademik membuat model pertama untuk memprediksi apakah mahasiswa lulus tepat waktu. Hasilnya mengecewakan.
Temukan masalah di data, perbaiki dengan langkah preprocessing yang tepat, lalu bangun model yang lebih baik.

**Cara memakai notebook ini**
1. Unggah file `data_akademik_kelulusan.csv` lewat panel **Files** (ikon folder di kiri) di Google Colab.
2. Jalankan sel dari atas ke bawah. Bagian bertanda **TODO** atau `___` harus Anda lengkapi.
3. Catat jawaban dan angka akurasi di lembar kerja kelompok.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

df = pd.read_csv("data_akademik_kelulusan.csv")
print(df.shape)
df.head()

## 1. Model pertama (sebelum preprocessing)
Ini cara bagian akademik membuat model pertama: semua kolom angka langsung dipakai, nilai kosong diisi 0, tanpa pembersihan dan tanpa scaling.

In [ ]:
kolom_angka = ["nim", "ipk_sem4", "sks_lulus", "kehadiran", "jam_belajar", "penghasilan_ortu"]
X0 = df[kolom_angka].fillna(0)
y0 = (df["lulus_tepat_waktu"] == "Ya").astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X0, y0, test_size=0.2, random_state=42, stratify=y0)
model0 = KNeighborsClassifier().fit(X_tr, y_tr)
print("Akurasi model pertama :", round(model0.score(X_te, y_te), 3))
print("Kalau selalu menebak 'Ya':", round(y_te.mean(), 3))

**Pertanyaan 1.** Bandingkan akurasi model pertama dengan akurasi kalau kita selalu menebak "Ya". Apa artinya?

## 2. Temukan masalah di data
Gunakan perintah eksplorasi untuk menemukan nilai kosong, data ganda, penulisan tidak konsisten, dan nilai yang tidak masuk akal.

In [ ]:
df.info()

In [ ]:
print("Jumlah nilai kosong per kolom:")
print(df.isna().sum())
print("\nJumlah baris ganda:", ___)  # TODO

In [ ]:
print(df["jenis_kelamin"].value_counts())
print(df["prodi"].value_counts())
print(df["kampus"].value_counts())

In [ ]:
df.describe().round(2)

**Pertanyaan 2.** Tuliskan minimal lima masalah yang Anda temukan, beserta kolomnya.

## 3. Data cleaning

In [ ]:
data = df.copy()

# 3a. Hapus baris ganda
data = ___  # TODO

# 3b. Seragamkan penulisan jenis kelamin dan prodi
peta_jk = {"l": "L", "laki-laki": "L", "laki": "L", "p": "P", "perempuan": "P"}
data["jenis_kelamin"] = data["jenis_kelamin"].str.strip().str.lower().map(peta_jk)
data["prodi"] = data["prodi"].str.strip().str.title()

print(len(data), "baris")
print(data["jenis_kelamin"].value_counts())

### Outlier: salah input atau memang nilai ekstrem?
Aturan IQR menandai nilai di bawah Q1 − 1,5·IQR atau di atas Q3 + 1,5·IQR. Tidak semua nilai yang ditandai harus dibuang.

In [ ]:
def cek_iqr(kolom):
    s = data[kolom].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    ditandai = s[(s < bawah) | (s > atas)]
    print(f"{kolom}: batas {bawah:.2f} sampai {atas:.2f}, ditandai {len(ditandai)} nilai -> {sorted(ditandai.tolist())}")

for k in ["ipk_sem4", "kehadiran", "jam_belajar"]:
    cek_iqr(k)

**Pertanyaan 3.** Dari nilai yang ditandai, mana yang jelas salah input dan mana yang masih masuk akal?

In [ ]:
# IPK di atas 4 adalah salah ketik (3.73 tertulis 37.3): bagi 10
salah_ipk = data["ipk_sem4"] > 4
data.loc[salah_ipk, "ipk_sem4"] = data.loc[salah_ipk, "ipk_sem4"] / 10

# Kehadiran di atas 100% dan jam belajar di atas 84 jam per minggu tidak mungkin: jadikan kosong
# TODO: tulis kode untuk menjadikan nilai yang tidak mungkin sebagai NaN

data[["ipk_sem4", "kehadiran", "jam_belajar"]].describe().round(2)

## 4. Feature selection
Tidak semua kolom layak dipakai. Perhatikan kolom yang hanya berisi ID, kolom yang nilainya sama semua, dan kolom yang baru diketahui setelah mahasiswa lulus.

In [ ]:
kolom_dibuang = ___  # TODO
kolom_numerik = ["ipk_sem4", "sks_lulus", "kehadiran", "jam_belajar", "penghasilan_ortu"]
kolom_nominal = ["jenis_kelamin", "prodi", "jalur_masuk", "ikut_organisasi"]
kolom_ordinal = ["kemampuan_bahasa"]

print("Kolom yang dibuang:", kolom_dibuang)
X = data[kolom_numerik + kolom_nominal + kolom_ordinal]
y = (data["lulus_tepat_waktu"] == "Ya").astype(int)
print(X.shape)

**Pertanyaan 4.** Jelaskan alasan setiap kolom yang dibuang. Mengapa `nilai_ta` berbahaya walaupun sangat berkaitan dengan label?

## 5. Train-test split
Bagi data **sebelum** imputasi, scaling, dan encoding, supaya informasi dari data test tidak bocor ke pelatihan.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=___)  # TODO: kolom apa untuk stratify?
print(X_train.shape, X_test.shape)

## 6. Pipeline preprocessing
`ColumnTransformer` menerapkan langkah berbeda untuk tiap jenis kolom. Semua langkah dipelajari dari data training saja saat `fit`.

In [ ]:
pra_numerik = Pipeline([
    ("imputasi", SimpleImputer(strategy="median")),
    ("scaling", ___),  # TODO
])
pra_nominal = Pipeline([
    ("imputasi", SimpleImputer(strategy="most_frequent")),
    ("encoding", ___),  # TODO
])
pra_ordinal = Pipeline([
    ("imputasi", SimpleImputer(strategy="most_frequent")),
    ("encoding", OrdinalEncoder(categories=[["Dasar", "Menengah", "Mahir"]])),
])
preprocessing = ColumnTransformer([
    ("num", pra_numerik, kolom_numerik),
    ("nom", pra_nominal, kolom_nominal),
    ("ord", pra_ordinal, kolom_ordinal),
])

## 7. Bangun model supervised sederhana

In [ ]:
hasil = {}
for nama, model in [("KNN", KNeighborsClassifier()),
                    ("Regresi logistik", LogisticRegression(max_iter=1000))]:
    pipe = Pipeline([("prep", preprocessing), ("model", model)])
    pipe.fit(X_train, y_train)
    hasil[nama] = round(pipe.score(X_test, y_test), 3)

hasil["Model pertama"] = round(model0.score(X_te, y_te), 3)
pd.Series(hasil, name="akurasi").sort_values(ascending=False)

**Pertanyaan 5.** Berapa kenaikan akurasi dibanding model pertama?

## 8. Eksperimen tambahan
### 8a. KNN tanpa scaling

In [ ]:
pra_tanpa_scaling = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), kolom_numerik),
    ("nom", pra_nominal, kolom_nominal),
    ("ord", pra_ordinal, kolom_ordinal),
])
knn_tanpa = Pipeline([("prep", pra_tanpa_scaling), ("model", KNeighborsClassifier())]).fit(X_train, y_train)
print("KNN tanpa scaling:", round(knn_tanpa.score(X_test, y_test), 3))

**Pertanyaan 6.** Mengapa KNN sangat terpengaruh oleh scaling? Kolom mana yang mendominasi perhitungan jarak?

### 8b. Bagaimana jika `nilai_ta` ikut dipakai?

In [ ]:
X_bocor = data[kolom_numerik + ["nilai_ta"] + kolom_nominal + kolom_ordinal]
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_bocor, y, test_size=0.2, random_state=42, stratify=y)
prep_bocor = ColumnTransformer([
    ("num", pra_numerik, kolom_numerik + ["nilai_ta"]),
    ("nom", pra_nominal, kolom_nominal),
    ("ord", pra_ordinal, kolom_ordinal),
])
model_bocor = Pipeline([("prep", prep_bocor), ("model", LogisticRegression(max_iter=1000))]).fit(Xb_tr, yb_tr)
print("Akurasi dengan nilai_ta:", round(model_bocor.score(Xb_te, yb_te), 3))

**Pertanyaan 7.** Akurasinya sangat tinggi. Mengapa hasil ini tidak bisa dipakai untuk memprediksi mahasiswa semester 4?

## Ringkasan untuk lembar kerja
| Langkah | Tindakan kelompok |
|---|---|
| Cleaning | |
| Scaling | |
| Encoding | |
| Feature yang dibuang | |
| Pembagian data | |
| Akurasi sebelum → sesudah | |